# CF-Nodule — Kaggle / Colab runner

Runs the whole pipeline on a free GPU. **No 125 GB download** — real data is pulled
for a few hundred patients over the public TCIA API (no login).

**How to use**
1. Kaggle: **Settings (right panel) → Accelerator = GPU**, and **Internet = ON**.
2. Add the code: right panel **+ Add Input → Upload → Dataset**, upload `cf_nodule.zip`.
   (On Colab instead, just run cell 1 and pick the zip when prompted.)
3. **Run All**. `MODE='demo'` shows results instantly. Switch to `MODE='lidc'` for real data.

## 1. Load the CF-Nodule code (works on Kaggle and Colab)

In [ ]:
import os, sys, glob, shutil, zipfile

def _has_src(d): return os.path.isdir(os.path.join(d, 'src'))

def find_repo():
    for c in ['cf_nodule', '/kaggle/working/cf_nodule', '.']:
        if _has_src(c): return c
    # a cf_nodule*.zip added as a Kaggle dataset or sitting in the working dir
    zips = glob.glob('/kaggle/input/**/cf_nodule*.zip', recursive=True) + \
           glob.glob('**/cf_nodule*.zip', recursive=True)
    dst = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
    if zips:
        print('unzipping', zips[0]); zipfile.ZipFile(zips[0]).extractall(dst)
    # a dataset folder that already contains src/ (zip extracted by Kaggle)
    for d in glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*'):
        if _has_src(d):
            tgt = os.path.join(dst, 'cf_nodule')
            if not _has_src(tgt): shutil.copytree(d, tgt, dirs_exist_ok=True)
    # Colab: prompt an upload if still nothing
    if not any(_has_src(c) for c in ['cf_nodule', '/kaggle/working/cf_nodule', '.']):
        try:
            from google.colab import files
            print('Upload cf_nodule.zip ...'); up = files.upload()
            zipfile.ZipFile(next(iter(up))).extractall('.')
        except Exception:
            pass
    for c in ['cf_nodule', '/kaggle/working/cf_nodule', '.']:
        if _has_src(c): return c
    raise SystemExit('Could not find the code. Upload cf_nodule.zip (Add Input) and re-run.')

repo = find_repo(); os.chdir(repo); sys.path.insert(0, os.getcwd())
print('cwd =', os.getcwd()); print('src/ ->', sorted(os.listdir('src')))

## 2. Install dependencies

In [ ]:
!pip -q install numpy scipy scikit-learn scikit-image matplotlib pyyaml tqdm pylidc pydicom SimpleITK captum requests
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 3. Choose a mode

In [ ]:
MODE = 'demo'      # 'demo' = instant synthetic proof  |  'lidc' = real data via TCIA API
N_PATIENTS = 60    # how many LIDC patients to fetch (start small; raise later)

## 4a. DEMO — runs in seconds, proves the method end-to-end

In [ ]:
if MODE == 'demo':
    !python demo/cf_demo.py --outdir demo/out
    from IPython.display import Image, display
    for f in ['fig1_attributions.png', 'fig3_metrics.png', 'fig4_risk_coverage.png']:
        display(Image('demo/out/' + f))

## 4b. REAL DATA — fetch a subset over the public TCIA API (no login)
Downloads DICOMs for `N_PATIENTS` only, registers them with pylidc, and builds a
small patch cache (image + mask + concepts + malignancy). Requires **Internet = ON**.

In [ ]:
if MODE == 'lidc':
    !python -m src.data.fetch_lidc_api --patients {N_PATIENTS} --out data/LIDC-IDRI
    !python -m src.data.download_lidc --set-path data/LIDC-IDRI --check
    !python -m src.data.lidc_dataset --build-cache --config configs/quick.yaml

## 5. Train the faithful model and its baseline  *(MODE='lidc')*

In [ ]:
if MODE == 'lidc':
    !python -m src.train --config configs/quick.yaml --cfa on
    !python -m src.train --config configs/quick.yaml --cfa off

## 6. Evaluate (faithfulness-first, paired stats) + figures  *(MODE='lidc')*

In [ ]:
if MODE == 'lidc':
    !python -m src.evaluate --config configs/quick.yaml \
        --ckpt runs/cfnodule_cfa-on/best.pt --baseline runs/cfnodule_cfa-off/best.pt --outdir runs/eval
    from IPython.display import Image, display; import json
    print(json.dumps(json.load(open('runs/eval/metrics.json')), indent=2))
    for f in ['risk_coverage.png', 'paired_effects.png']:
        p = 'runs/eval/' + f
        if os.path.exists(p): display(Image(p))

## Full paper experiments (optional)
For the complete results (patient-level 5-fold CV, CBM/CEM, cross-scanner, multi-seed),
fetch more patients (e.g. `N_PATIENTS = 300`), then run the scripts in the repo root:
`kfold.py`, `cem.py`, `cross_scanner.py`, `multiseed.py`, `final_main.py`. See `README.md`.